# Testes do artigo: risco de reidentificação em populações pequenas

Este notebook reproduz, em ordem, todos os experimentos do artigo *Quando a anonimização não basta* e mostra as tabelas e figuras correspondentes. Ele apenas chama os scripts do repositório, cada um num processo separado, exatamente como no terminal; os números saem idênticos aos do artigo, porque a semente aleatória é fixa (2026).

**Antes de começar**

- **No computador:** este arquivo precisa estar na mesma pasta dos scripts (`sim_reidentificacao.py`, `rede_reidentificacao.py` etc.), e o kernel do Jupyter precisa ser um Python **3.10 a 3.14** com os pacotes do `requirements.txt` (o pgmpy ainda não funciona no 3.15). A célula 0 confere isso.
- Rode as células de cima para baixo (**Shift+Enter**). Enquanto uma célula roda, aparece `[*]` à esquerda; quando termina, aparece um número.
- Tempo total: cerca de 10 minutos. As células mais demoradas estão indicadas.

Todos os dados são sintéticos e gerados pelo próprio código: nenhum dado pessoal é coletado, lido ou vinculado a pessoas.

**No Google Colab:** basta rodar as células em ordem (*Ambiente de execução → Executar tudo*). A primeira célula baixa o repositório do GitHub e instala o pgmpy; no computador local, ela não faz nada.

## Preparação (Google Colab)

In [ ]:
import sys, os
if "google.colab" in sys.modules:
    if not os.getcwd().endswith("reidentificacao-populacoes-pequenas"):
        if not os.path.exists("reidentificacao-populacoes-pequenas"):
            !git clone -q https://github.com/barbozapauloricardo/reidentificacao-populacoes-pequenas
        %cd reidentificacao-populacoes-pequenas
    !pip install -q "pgmpy>=1.1"
    print("Repositório baixado e pacotes instalados.")
else:
    print("Execução local: nada a preparar.")

## 0. Conferência do ambiente

In [ ]:
import sys, platform
from pathlib import Path
import numpy as np, pandas as pd, matplotlib, scipy, pgmpy
from IPython.display import display, Image, Markdown

print("Python    ", sys.version.split()[0], "|", platform.system(), platform.release())
print("numpy     ", np.__version__)
print("pandas    ", pd.__version__)
print("matplotlib", matplotlib.__version__)
print("scipy     ", scipy.__version__)
print("pgmpy     ", pgmpy.__version__)
assert Path("sim_reidentificacao.py").exists(), "Coloque este notebook na pasta dos scripts."
assert sys.version_info[:2] < (3, 15), "Use um kernel Python 3.10 a 3.14 (o pgmpy 1.1 não roda no 3.15)."
pd.set_option("display.precision", 3)

import os, subprocess, time
def rodar(script):
    # roda o script num processo separado, igual a rodá-lo no terminal
    t0 = time.perf_counter()
    env = dict(os.environ, PYTHONIOENCODING="utf-8", MPLBACKEND="Agg")
    r = subprocess.run([sys.executable, "-W", "ignore::FutureWarning", script],
                       capture_output=True, text=True, encoding="utf-8", errors="replace", env=env)
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr)
        raise RuntimeError(f"{script} terminou com erro (veja a mensagem acima).")
    print(f"[{script} concluído em {time.perf_counter() - t0:.0f} s]")

print("\nAmbiente OK.")

## Os dados sintéticos

Os experimentos geram seus próprios dados a partir da rede bayesiana geradora. A célula abaixo grava dois exemplos na pasta `dados_sinteticos/` para inspeção: uma turma de 30 respondentes, com o tamanho da classe de equivalência de cada registro sob os desenhos D0 a D3 (colunas `F_D0` a `F_D3`; 1 = respondente único), e uma população de 5.000 respondentes.

In [ ]:
import subprocess as _sp, sys as _sys, os as _os
_sp.run([_sys.executable, "gerar_dados_exemplo.py"], capture_output=True,
        env=dict(_os.environ, PYTHONIOENCODING="utf-8"))
turma = pd.read_csv("dados_sinteticos/turma_N30.csv")
display(turma)
print("Respondentes únicos nesta turma:",
      {d: f"{(turma[f'F_{d}'] == 1).mean():.0%}" for d in ["D0", "D1", "D2", "D3"]})

## 1. Simulação da unicidade (E1 a E3)

A rede geradora produz turmas sintéticas com sete quase-identificadores. Para cada desenho de questionário (D0 a D3), mede-se a proporção de respondentes únicos por simulação de Monte Carlo e por expressões analíticas.

⏱ **Esta célula leva de 3 a 5 minutos e não mostra nada até terminar.**

In [ ]:
rodar("sim_reidentificacao.py")

**E1. Respondentes únicos por tamanho da turma** (Monte Carlo × expressão analítica)

In [ ]:
e1 = pd.read_csv("resultados/E1_tamanho_turma.csv")
tab = e1.pivot(index="N", columns="desenho", values="prop_unicos_MC") * 100
display(tab.round(1).style.format("{:.1f}%").set_caption("Únicos (%) por N e desenho"))
dif = (e1.prop_unicos_MC - e1.prop_unicos_analitico).abs().max() * 100
print(f"Diferença máxima entre Monte Carlo e expressão analítica: {dif:.2f} ponto percentual")

**E2. Acúmulo de quase-identificadores** (N = 30, desenho D0)

In [ ]:
e2 = pd.read_csv("resultados/E2_numero_qi.csv")
e2["únicos (%)"] = (100 * e2.prop_unicos).round(1)
e2["k < 5 (%)"] = (100 * e2.prop_k_menor_5).round(1)
display(e2[["num_QI", "QI_incluidos", "únicos (%)", "k < 5 (%)"]])

**E3. Resposta incompleta** (60% de adesão, N = 30)

In [ ]:
e3 = pd.read_csv("resultados/E3_resposta_incompleta.csv")
display((e3.set_index("desenho")[["respondentes", "prop_unicos_na_turma",
         "precisao_unico_amostra", "risco_medio_respondentes"]]).round(3))

## 2. Sensibilidade aos parâmetros (E4)

Quatro cenários de composição da turma, do heterogêneo (S1) ao muito homogêneo (S3). ⏱ Cerca de 30 segundos.

In [ ]:
rodar("sensibilidade.py")

In [ ]:
e4 = pd.read_csv("resultados/E4_sensibilidade.csv")
t = e4[e4.N == 30].pivot(index="desenho", columns="cenario", values="prop_unicos") * 100
display(t.round(1).style.format("{:.1f}%").set_caption("Únicos (%) com N = 30"))

## 3. Rede bayesiana de reidentificação (E5 e E6)

Rede: desenho (*D*), tamanho da turma (*N*) e composição (*S*) → classe de equivalência (*F*); *F* e tipo de adversário (*K*) → reidentificação (*Re*). A tabela de *F* é calculada a partir da rede geradora. A célula faz inferência preditiva e diagnóstica, verifica d-separações, compara observar e intervir e busca a explicação mais relevante (MRE).

⏱ **Cerca de 3 a 4 minutos.**

In [ ]:
rodar("rede_reidentificacao.py")

**Tabela 4 do artigo:** P(Re = sim | D, N, K = interno)

In [ ]:
a1 = pd.read_csv("resultados/A1_preditiva.csv")
display((a1.pivot(index="D", columns="N", values="P_Re_sim") * 100).round(1)
        .style.format("{:.1f}%"))

**Inferência diagnóstica:** dado Re = sim numa turma de 30

In [ ]:
a2 = pd.read_csv("resultados/A2_diagnostica.csv")
display(a2.round(3))

**Tabela 5 do artigo:** observar × intervir (variante com confundimento N → D)

In [ ]:
a4 = pd.read_csv("resultados/A4_observar_intervir.csv").set_index("D")
display((a4 * 100).round(1).rename(columns={"P_Re_obs": "P(Re | D)", "P_Re_do": "P(Re | do(D))"}))
print(f"Diferença D0 − D3 ao observar: {100*(a4.P_Re_obs['D0']-a4.P_Re_obs['D3']):.1f} p.p.")
print(f"Diferença D0 − D3 ao intervir: {100*(a4.P_Re_do['D0']-a4.P_Re_do['D3']):.1f} p.p.")

**Explicação mais relevante (MRE):** explicações mínimas dado Re = sim

In [ ]:
display(pd.read_csv("resultados/A5_mre_minimas.csv").round(3))
display(Markdown("Com o adversário interno fixado (busca sobre D, N, S):"))
display(pd.read_csv("resultados/A5_mre_interno_minimas.csv").round(3))

## 4. Sensibilidade da MRE às hipóteses arbitradas

Refaz a MRE variando P(K = interno) e a chance de o adversário externo vincular um registro único. ⏱ Cerca de 1 a 2 minutos.

In [ ]:
rodar("sensibilidade_mre.py")

In [ ]:
a6 = pd.read_csv("resultados/A6_sensibilidade_mre.csv")
display(a6.pivot(index="P_externo_unico", columns="P_K_interno", values="melhor_explicacao"))

## 5. Aprendizado de estrutura (E7)

Com quantos respondentes os algoritmos PC e Hill Climbing (BIC) recuperam a estrutura da rede geradora? ⏱ Cerca de 1 minuto.

In [ ]:
rodar("aprendizado_estrutura.py")

## 6. Figuras do artigo

In [ ]:
rodar("figuras_artigo.py")
for f in ["fig_E1_tamanho", "fig_E2_numero_qi", "fig_E4_sensibilidade", "fig_E7_estrutura"]:
    display(Markdown(f"**{f}**"))
    display(Image(filename=f"figuras/{f}.png", width=480))

## 7. Tempos de execução

In [ ]:
print(open("resultados/ambiente.txt", encoding="utf-8").read())

## 8. Experimente você mesmo

A rede de reidentificação já está pronta na memória. Mude os valores abaixo e rode a célula para ver como a probabilidade de reidentificação responde. Isso não altera os resultados do artigo.

- `desenho`: "D0", "D1", "D2" ou "D3"
- `tamanho`: "15", "30", "100" ou "500" (entre aspas)
- `adversario`: "interno" ou "externo"

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
from pgmpy.inference import VariableElimination
import rede_reidentificacao as rr

desenho, tamanho, adversario = "D3", "30", "interno"

if "rede" not in globals():
    rede = rr.montar_rede()          # calcula a CPT de F (cerca de 1 minuto na primeira vez)
ve = VariableElimination(rede)
q = ve.query(["Re"], evidence={"D": desenho, "N": tamanho, "K": adversario}, show_progress=False)
print(f"P(Re = sim | D={desenho}, N={tamanho}, K={adversario}) = {rr.p(q, 'sim'):.3f}")